# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pangakeerthi11/keerthi-flyrank-ml-internship-2026/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Method Choice and Why

I will start with a Decision Tree Classifier for the modeling task. A decision tree learns patterns from input features and uses them to predict a class.

I chose this method because it is relatively easy to understand and interpret. I will use only information available at prediction time and avoid future information, target-derived columns, and existing product flags as model inputs.

I will compare the model with my Week-4 rule-based baseline using the same eligible records, evaluation split, and metric. The goal is to check whether the model provides a useful improvement rather than assuming that a more complex method is better.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Split Design

I will inspect the available client identifiers, observation dates, target definition, and data grain before choosing the validation split. The split must prevent information from the evaluation records leaking into model training.

If records from the same client could be related, I will consider a client-grouped split. If the task is to predict future outcomes, I will consider a time-aware split. I will use the split that best reflects the intended prediction setting and apply the same evaluation records to the model and baseline.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
import json
import pandas as pd
from pathlib import Path

with open("outputs/model_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100",
    "average_precision",
    "roc_auc",
]

rows = {}

baseline = results["baseline"]
rows["Week-4 Baseline"] = {
    metric: baseline.get(f"baseline_{metric}")
    for metric in metrics
}

for name, values in results["models"].items():
    rows[name.replace("_", " ").title()] = {
        metric: values.get(metric)
        for metric in metrics
    }

comparison = pd.DataFrame.from_dict(rows, orient="index")
display(comparison.round(4))

print("Best model:", results["best_model"]["name"])
print("Split strategy:", results["split_strategy"])
print("Training rows:", results["train_rows"])
print("Testing rows:", results["test_rows"])

,accuracy,precision,recall,f1,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc
Week-4 Baseline,0.6086,0.4986,0.1892,0.2743,0.15,0.24,0.36,0.4676,0.6269
Decision Tree,0.6766,0.5686,0.7162,0.6339,0.45,0.58,0.62,0.5753,0.7415
Logistic Regression,0.6606,0.5659,0.5666,0.5662,0.35,0.40,0.44,0.5215,0.7003
Random Forest,0.6723,0.5610,0.7437,0.6395,0.65,0.74,0.72,0.6182,0.7500


Best model: random_forest
Split strategy: client_holdout
Training rows: 27675
Testing rows: 2325


In [22]:
import json
from pathlib import Path

path = Path("outputs/model_results.json")
results = json.loads(path.read_text(encoding="utf-8"))

print("MODEL RESULTS:")
print(json.dumps(results.get("models", {}), indent=2))

print("\nBASELINE RESULTS:")
print(json.dumps(results.get("baseline", {}), indent=2))

MODEL RESULTS:
{
  "decision_tree": {
    "accuracy": 0.6765591397849462,
    "average_precision": 0.5753189989001579,
    "f1": 0.6338851022395326,
    "precision": 0.5685589519650655,
    "precision_at_100": 0.62,
    "precision_at_20": 0.45,
    "precision_at_50": 0.58,
    "recall": 0.7161716171617162,
    "roc_auc": 0.7415203737887913
  },
  "logistic_regression": {
    "accuracy": 0.6606451612903226,
    "average_precision": 0.5215418768632023,
    "f1": 0.5662451896646509,
    "precision": 0.5659340659340659,
    "precision_at_100": 0.44,
    "precision_at_20": 0.35,
    "precision_at_50": 0.4,
    "recall": 0.5665566556655666,
    "roc_auc": 0.7002914980763613
  },
  "random_forest": {
    "accuracy": 0.672258064516129,
    "average_precision": 0.6182189675945406,
    "f1": 0.6395458845789972,
    "precision": 0.5609958506224066,
    "precision_at_100": 0.72,
    "precision_at_20": 0.65,
    "precision_at_50": 0.74,
    "recall": 0.7436743674367436,
    "roc_auc": 0.75002952272

I will use the official FlyRank training pipeline to predict `is_declining_label`, which indicates whether content is declining. I will compare Logistic Regression, Decision Tree, and Random Forest models against the Week-4 rule-based baseline.

The pipeline uses a client-aware train/test split when possible, keeping records from the same client out of both sets. It evaluates the models and baseline on the same test records using Precision@50, average precision, ROC-AUC, and other classification metrics. The best model is selected primarily using Precision@50.

I will present the actual evaluation results in a comparison table and use the results to determine whether a machine-learning model improves on the baseline. I will also report the important features and acknowledge any limitations.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

repo = Path("/content/keerthi-flyrank-ml-internship-2026")
predictions_path = repo / "data/processed/model_predictions.csv"

if not predictions_path.exists():
    raise FileNotFoundError(
        f"Predictions file not found: {predictions_path}"
    )

predictions = pd.read_csv(predictions_path)

# Use only held-out test records
test_data = predictions[predictions["split"] == "test"].copy()

# Actual labels and predicted labels from the selected model
y_test = test_data["is_declining_label"].astype(int)
y_pred = (
    test_data["best_model_probability"] >= 0.5
).astype(int)

print("Best model:", test_data["best_model_name"].iloc[0])
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

tn, fp, fn, tp = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)


Best model: random_forest

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.82      1416
           1       0.72      0.67      0.70       909

    accuracy                           0.77      2325
   macro avg       0.76      0.75      0.76      2325
weighted avg       0.77      0.77      0.77      2325

True negatives: 1180
False positives: 236
False negatives: 296
True positives: 613


### Error Analysis and Interpretation

The Random Forest model achieved an accuracy of **77%** on 2,325 test records. For class 1 (declining content), precision was 0.72 and recall was 0.67.

The model correctly identified 613 declining records (true positives) and 1,180 non-declining records (true negatives). It made 236 false-positive predictions and 296 false-negative predictions.

False positives are records predicted as declining even though their actual label was non-declining. False negatives are declining records that the model failed to identify. These errors show that the model does not identify every declining record correctly.

The results are based on the held-out test records and do not guarantee the same performance on future data. Further investigation of misclassified records and feature importance can help identify possible weaknesses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.